# PotholeRisk: Feature Engineering

This notebook creates engineered features from the cleaned Bengaluru accident dataset and traffic dataset.

These features will improve prediction performance and serve as inputs to the proposed Adaptive Dynamic Bayesian Network.

In [2]:
import pandas as pd
import numpy as np

In [3]:
accident_df = pd.read_csv(
    "../dataset/processed/bengaluru_accidents_clean.csv"
)

traffic_df = pd.read_csv(
    "../dataset/traffic/Banglore_traffic_Dataset.csv"
)

print(accident_df.shape)
print(traffic_df.shape)

(49932, 26)
(8936, 16)


In [4]:
import os

print(os.path.exists("../dataset/processed/bengaluru_accidents_clean.csv"))

True


## Feature 1: Weather Severity Index

The weather categories are converted into numerical risk scores based on how dangerous they are for road travel.

Higher scores indicate more hazardous driving conditions.

In [5]:
weather_scores = {

    "Clear": 0,
    "Fine": 0,

    "Cloudy": 1,

    "Fog / Mist": 2,
    "Mist or Fog": 2,

    "Light Rain": 3,

    "Heavy Rain": 4,

    "Wind": 2,
    "Strong Wind": 3,

    "Very Cold": 2,
    "Very Hot": 1,

    "Dust Storm": 4,

    "Flooding of Slipways/Rivulets": 5,

    "Snow": 5,

    "Hail or Sleet": 5,

    "Others": 2,

    "Not Applicable": 1
}

accident_df["Weather_Severity"] = accident_df["Weather"].map(weather_scores)

In [6]:
accident_df[
    ["Weather", "Weather_Severity"]
].drop_duplicates().sort_values("Weather_Severity")

,Weather,Weather_Severity
0,Clear,0.0
1,Fine,0.0
504,Very Hot,1.0
313,Cloudy,1.0
1376,Not Applicable,1.0
115,Others,2.0
612,Fog / Mist,2.0
1463,Very Cold,2.0
2855,Wind,2.0
2799,Mist or Fog,2.0


In [7]:
print("Road Condition")
print(accident_df["Road_Condition"].value_counts())

print("\nSurface Condition")
print(accident_df["Surface_Condition"].value_counts())

print("\nSurface Type")
print(accident_df["Surface_Type"].value_counts())

Road Condition
Road_Condition
Not Applicable                  33195
No influence on accident        15400
Construction Work / Material      423
Engineering Defect of Road        362
Pot holed                         354
Drainage Ditch                    198
Name: count, dtype: int64

Surface Condition
Surface_Condition
Dry                  41611
Others                4394
Not Applicable        3267
Wet                    223
Ditch or Potholed      183
Muddy                  163
Flooded                 91
Name: count, dtype: int64

Surface Type
Surface_Type
Bitumen(Tar)      43777
Not Applicable     3197
Concrete           1584
Surfaced            839
Kutcha              356
Gravel              148
Metalled             31
Name: count, dtype: int64


In [8]:
# Road Condition Score
road_condition_scores = {
    "No influence on accident": 0,
    "Not Applicable": 0,
    "Construction Work / Material": 2,
    "Engineering Defect of Road": 4,
    "Pot holed": 5,
    "Drainage Ditch": 4
}

# Surface Condition Score
surface_condition_scores = {
    "Dry": 0,
    "Not Applicable": 0,
    "Others": 1,
    "Wet": 3,
    "Muddy": 4,
    "Ditch or Potholed": 5,
    "Flooded": 5
}

# Surface Type Score
surface_type_scores = {
    "Bitumen(Tar)": 0,
    "Concrete": 0,
    "Cement Concrete": 0,
    "Not Applicable": 0,
    "Kutcha": 4,
    "Gravel": 3,
    "Metalled": 2
}

In [9]:
accident_df["Road_Condition_Score"] = accident_df["Road_Condition"].map(road_condition_scores)

accident_df["Surface_Condition_Score"] = accident_df["Surface_Condition"].map(surface_condition_scores)

accident_df["Surface_Type_Score"] = accident_df["Surface_Type"].map(surface_type_scores)    

In [10]:
accident_df["Road_Risk_Index"] = (
    accident_df["Road_Condition_Score"] +
    accident_df["Surface_Condition_Score"] +
    accident_df["Surface_Type_Score"]
)

In [11]:
accident_df[
    [
        "Road_Condition",
        "Surface_Condition",
        "Surface_Type",
        "Road_Risk_Index"
    ]
].head(20)

,Road_Condition,Surface_Condition,Surface_Type,Road_Risk_Index
0,Not Applicable,Others,Bitumen(Tar),1.0
1,Not Applicable,Dry,Bitumen(Tar),0.0
2,Not Applicable,Dry,Bitumen(Tar),0.0
3,Not Applicable,Dry,Bitumen(Tar),0.0
4,Not Applicable,Dry,Bitumen(Tar),0.0
5,Not Applicable,Dry,Bitumen(Tar),0.0
6,Not Applicable,Not Applicable,Bitumen(Tar),0.0
7,Not Applicable,Wet,Bitumen(Tar),3.0
8,Not Applicable,Dry,Bitumen(Tar),0.0
9,Not Applicable,Dry,Bitumen(Tar),0.0


In [12]:
traffic_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8936 entries, 0 to 8935
Data columns (total 16 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Date                                8936 non-null   str    
 1   Area Name                           8936 non-null   str    
 2   Road/Intersection Name              8936 non-null   str    
 3   Traffic Volume                      8936 non-null   int64  
 4   Average Speed                       8936 non-null   float64
 5   Travel Time Index                   8936 non-null   float64
 6   Congestion Level                    8936 non-null   float64
 7   Road Capacity Utilization           8936 non-null   float64
 8   Incident Reports                    8936 non-null   int64  
 9   Environmental Impact                8936 non-null   float64
 10  Public Transport Usage              8936 non-null   float64
 11  Traffic Signal Compliance           8936 non-null   fl

In [13]:
traffic_df[
    [
        "Traffic Volume",
        "Average Speed",
        "Congestion Level",
        "Travel Time Index",
        "Road Capacity Utilization"
    ]
].head(15)

,Traffic Volume,Average Speed,Congestion Level,Travel Time Index,Road Capacity Utilization
0,50590,50.230299,100.000000,1.500000,100.000000
1,30825,29.377125,100.000000,1.500000,100.000000
2,7399,54.474398,28.347994,1.039069,36.396525
3,60874,43.817610,100.000000,1.500000,100.000000
4,57292,41.116763,100.000000,1.500000,100.000000
5,47848,34.241963,100.000000,1.500000,100.000000
6,36574,29.982430,100.000000,1.500000,100.000000
7,25379,38.455179,79.038823,1.500000,100.000000
8,25022,35.039373,78.979596,1.500000,100.000000
9,31760,56.904556,97.672462,1.500000,100.000000


In [14]:
from sklearn.preprocessing import MinMaxScaler

In [15]:
scaler = MinMaxScaler()

traffic_features = [
    "Traffic Volume",
    "Average Speed",
    "Congestion Level",
    "Travel Time Index",
    "Road Capacity Utilization"
]

traffic_df[traffic_features] = scaler.fit_transform(
    traffic_df[traffic_features]
)

In [16]:
traffic_df[traffic_features].describe()

,Traffic Volume,Average Speed,Congestion Level,Travel Time Index,Road Capacity Utilization
count,8936.000000,8936.000000,8936.000000,8936.000000,8936.000000
mean,0.368744,0.278653,0.797743,0.751088,0.901910
std,0.191750,0.153419,0.248136,0.330664,0.204077
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.223874,0.168730,0.623501,0.484879,0.967450
50%,0.344616,0.275099,0.919749,1.000000,1.000000
75%,0.498857,0.381777,1.000000,1.000000,1.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000


In [17]:
traffic_df["Traffic_Exposure_Index"] = (
      0.30 * traffic_df["Traffic Volume"]
    + 0.25 * traffic_df["Congestion Level"]
    + 0.20 * traffic_df["Travel Time Index"]
    + 0.15 * traffic_df["Road Capacity Utilization"]
    + 0.10 * (1 - traffic_df["Average Speed"])
)

In [18]:
traffic_df[
    [
        "Traffic Volume",
        "Average Speed",
        "Congestion Level",
        "Travel Time Index",
        "Road Capacity Utilization",
        "Traffic_Exposure_Index"
    ]
].head(15)

,Traffic Volume,Average Speed,Congestion Level,Travel Time Index,Road Capacity Utilization,Traffic_Exposure_Index
0,0.683671,0.433156,1.000000,1.000000,1.000000,0.861786
1,0.392178,0.134360,1.000000,1.000000,1.000000,0.804217
2,0.046692,0.493967,0.244494,0.078067,0.217287,0.173941
3,0.835339,0.341271,1.000000,1.000000,1.000000,0.916475
4,0.782512,0.302572,1.000000,1.000000,1.000000,0.904496
5,0.643232,0.204066,1.000000,1.000000,1.000000,0.872563
6,0.476964,0.143034,1.000000,1.000000,1.000000,0.828786
7,0.311860,0.264436,0.778983,1.000000,1.000000,0.711860
8,0.306595,0.215492,0.778359,1.000000,1.000000,0.715019
9,0.405967,0.528788,0.975458,1.000000,1.000000,0.762776
